# HQDE v0.1.10 Benchmark - CIFAR-10

**Focused benchmark using the new HQDE v0.1.10 API with:**
- SmallImageResNet18 from package
- make_cifar_training_config() preset
- New evaluate() method
- Validation tracking during training

**Target:** Match or exceed 88.45% accuracy from notebook baseline

In [ ]:
# Install dependencies
# Install HQDE v0.1.10 directly from GitHub (includes SmallImageResNet18 and new API)
%pip install -q torch torchvision
%pip install -q git+https://github.com/Prathmesh333/HQDE-PyPI.git

In [ ]:
import time
import numpy as np
import torch
import torchvision
from torch.utils.data import DataLoader

# Import from HQDE package
from hqde import (
    create_hqde_system,
    SmallImageResNet18,
    make_cifar_training_config,
    DataPreprocessor,
)

print(f'PyTorch: {torch.__version__}')
print(f'Device: {"CUDA" if torch.cuda.is_available() else "CPU"}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} ({props.total_memory/1024**3:.1f} GB)')

## Configuration

In [ ]:
# Experiment settings
DATASET = 'cifar10'
NUM_WORKERS = 4
NUM_EPOCHS = 20
BATCH_SIZE = 128
SEED = 42

# Set seed
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## Load CIFAR-10 Dataset

In [ ]:
# Use HQDE's data preprocessor
train_transform = DataPreprocessor.cifar10_transforms(is_training=True)
test_transform = DataPreprocessor.cifar10_transforms(is_training=False)

# Load datasets
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=test_transform
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples: {len(test_dataset)}')

## Create HQDE System with New API

In [ ]:
# Use the new training preset
training_config = make_cifar_training_config(
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
)

print('Training Configuration:')
for key, value in training_config.items():
    print(f'  {key}: {value}')

# Create HQDE system with SmallImageResNet18
system = create_hqde_system(
    model_class=SmallImageResNet18,
    model_kwargs={'num_classes': 10},
    num_workers=NUM_WORKERS,
    training_config=training_config,
)

print(f'\nHQDE System created with {NUM_WORKERS} workers')
print(f'Model: SmallImageResNet18')
print(f'Ensemble mode: {training_config["ensemble_mode"]}')
print(f'Batch assignment: {training_config["batch_assignment"]}')

## Train with Validation Tracking

In [ ]:
print(f'Starting training for {NUM_EPOCHS} epochs...\n')
start_time = time.time()

# NEW: train() now accepts validation_loader and tracks val metrics
metrics = system.train(
    train_loader,
    num_epochs=NUM_EPOCHS,
    validation_loader=test_loader,  # NEW in v0.1.9!
)

training_time = time.time() - start_time

print(f'\nTraining completed in {training_time:.2f} seconds')
print(f'Average time per epoch: {training_time/NUM_EPOCHS:.2f} seconds')

## Display Training History

In [ ]:
import pandas as pd

# Extract epoch history
epoch_history = metrics.get('epoch_history', [])

if epoch_history:
    df = pd.DataFrame(epoch_history)
    print('\nTraining History:')
    print(df.to_string(index=False))
    
    # Show final metrics
    final = epoch_history[-1]
    print(f'\nFinal Results:')
    print(f'  Train Loss: {final["loss"]:.4f}')
    print(f'  Train Accuracy: {final["accuracy"]:.4f}')
    if 'val_loss' in final:
        print(f'  Val Loss: {final["val_loss"]:.4f}')
        print(f'  Val Accuracy: {final["val_accuracy"]:.4f}')
else:
    print('No epoch history available')

## Evaluate on Test Set

In [ ]:
# NEW: Use the evaluate() method
print('Evaluating on test set...')
test_metrics = system.evaluate(test_loader)

print(f'\nTest Set Results:')
print(f'  Loss: {test_metrics["loss"]:.4f}')
print(f'  Accuracy: {test_metrics["accuracy"]:.4f} ({test_metrics["accuracy"]*100:.2f}%)')

# Compare to target
target_accuracy = 0.8845
print(f'\nTarget Accuracy: {target_accuracy:.4f} ({target_accuracy*100:.2f}%)')
if test_metrics['accuracy'] >= target_accuracy:
    print('✅ Target achieved!')
else:
    diff = target_accuracy - test_metrics['accuracy']
    print(f'❌ Below target by {diff:.4f} ({diff*100:.2f}%)')

## Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

if epoch_history:
    epochs = [h['epoch'] for h in epoch_history]
    train_loss = [h['loss'] for h in epoch_history]
    train_acc = [h['accuracy'] for h in epoch_history]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss plot
    ax1.plot(epochs, train_loss, 'b-', label='Train Loss', linewidth=2)
    if 'val_loss' in epoch_history[0]:
        val_loss = [h['val_loss'] for h in epoch_history]
        ax1.plot(epochs, val_loss, 'r-', label='Val Loss', linewidth=2)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Accuracy plot
    ax2.plot(epochs, train_acc, 'b-', label='Train Accuracy', linewidth=2)
    if 'val_accuracy' in epoch_history[0]:
        val_acc = [h['val_accuracy'] for h in epoch_history]
        ax2.plot(epochs, val_acc, 'r-', label='Val Accuracy', linewidth=2)
    ax2.axhline(y=target_accuracy, color='g', linestyle='--', label='Target (88.45%)', linewidth=2)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title('Training Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print('No data to plot')

## Summary

In [ ]:
print('='*60)
print('HQDE v0.1.10 Benchmark Summary')
print('='*60)
print(f'Dataset: CIFAR-10')
print(f'Model: SmallImageResNet18')
print(f'Workers: {NUM_WORKERS}')
print(f'Epochs: {NUM_EPOCHS}')
print(f'Batch Size: {BATCH_SIZE}')
print(f'\nTraining Time: {training_time:.2f}s ({training_time/NUM_EPOCHS:.2f}s/epoch)')
print(f'\nFinal Test Accuracy: {test_metrics["accuracy"]:.4f} ({test_metrics["accuracy"]*100:.2f}%)')
print(f'Target Accuracy: {target_accuracy:.4f} ({target_accuracy*100:.2f}%)')
print(f'\nStatus: {"✅ Target Achieved" if test_metrics["accuracy"] >= target_accuracy else "❌ Below Target"}')
print('='*60)

## Cleanup

In [ ]:
# Clean up resources
system.cleanup()
print('Resources cleaned up')